The following is the code needed to download the data we used in this assignment. <br>
To run it, log into WRDS, then go to PROGRAMS > JupyterHub > Open JupyterHub > New Launcher and make a notebook with a python 3 kernal. <br>
Finally, paste the below code in, and it should run. You may need to change the username in the line "db = wrds.Connection(wrds_username="nzanatta")"

In [6]:
import wrds
import pandas as pd
from pathlib import Path
from IPython.display import display, FileLink

# ==========================================
# 1. Settings
# ==========================================

START_DATE = "2000-01-01"
END_DATE = "2026-01-01"

# Save all files in the current working directory
OUTPUT_DIR = Path.cwd()

print(f"Files will be saved to: {OUTPUT_DIR}")

db = wrds.Connection(wrds_username="nzanatta")

# ==========================================
# 2. S&P 500 stock returns
# ==========================================

stocks_df = db.raw_sql(f"""
    SELECT
        s.mthcaldt AS date,
        s.permno,
        s.permco,
        s.ticker,
        s.issuernm AS company_name,
        ABS(s.mthprc) AS price,
        s.mthret AS total_return

    FROM crsp_a_stock.msf_v2 AS s

    INNER JOIN crsp_a_indexes.msp500list_v2 AS c
        ON s.permno = c.permno
        AND s.mthcaldt BETWEEN c.mbrstartdt
        AND COALESCE(
            c.mbrenddt,
            CAST('9999-12-31' AS date)
        )

    WHERE s.mthcaldt >= '{START_DATE}'
      AND s.mthcaldt < '{END_DATE}'

    ORDER BY s.mthcaldt, s.permno
""", date_cols=["date"])

# ==========================================
# 3. Market returns and risk-free rate
# Source: Fama-French monthly factors
# ==========================================

factors_df = db.raw_sql(f"""
    SELECT
        dateff AS date,
        mktrf,
        rf

    FROM ff_all.factors_monthly

    WHERE dateff >= '{START_DATE}'
      AND dateff < '{END_DATE}'

    ORDER BY dateff
""", date_cols=["date"])

db.close()

# ==========================================
# 4. Create separate market and RF files
# ==========================================

# Total market return = market excess return + RF
market_df = factors_df[["date"]].copy()

market_df["market_return"] = (
    factors_df["mktrf"].to_numpy()
    + factors_df["rf"].to_numpy()
)

# Risk-free rate
rf_df = factors_df[["date", "rf"]].copy()

rf_df = rf_df.rename(
    columns={"rf": "rf_1m"}
)

# ==========================================
# 5. Standardize dates to month-end
# ==========================================

stocks_df = stocks_df.copy()
market_df = market_df.copy()
rf_df = rf_df.copy()

stocks_df["date"] = (
    pd.to_datetime(stocks_df["date"])
    .dt.to_period("M")
    .dt.to_timestamp("M")
)

market_df["date"] = (
    pd.to_datetime(market_df["date"])
    .dt.to_period("M")
    .dt.to_timestamp("M")
)

rf_df["date"] = (
    pd.to_datetime(rf_df["date"])
    .dt.to_period("M")
    .dt.to_timestamp("M")
)

# ==========================================
# 6. Save three CSV files
# ==========================================

files = {
    "sp500_stocks_2000_2025.csv": stocks_df,
    "market_returns_2000_2025.csv": market_df,
    "risk_free_rate_2000_2025.csv": rf_df
}

for filename, data in files.items():

    if data.empty:
        print(f"WARNING: {filename} has no data.")
        continue

    output_path = OUTPUT_DIR / filename

    data.to_csv(output_path, index=False)

    print(f"\nSaved: {output_path}")
    print(f"Rows: {len(data):,}")

    display(FileLink(str(output_path)))

# ==========================================
# 7. Display previews
# ==========================================

print("\n========== S&P 500 STOCKS ==========")
display(stocks_df.head(10))

print("\n========== MARKET RETURNS ==========")
display(market_df.head(10))

print("\n========== RISK-FREE RATE ==========")
display(rf_df.head(10))

# ==========================================
# 8. Summary
# ==========================================

print("\n========== SUMMARY ==========")

print(f"Stock observations: {len(stocks_df):,}")

print(
    f"Unique securities (PERMNO): "
    f"{stocks_df['permno'].nunique():,}"
)

print(
    f"Unique companies (PERMCO): "
    f"{stocks_df['permco'].nunique():,}"
)

print(
    f"Market observations: {len(market_df):,}"
)

print(
    f"Risk-free observations: {len(rf_df):,}"
)

print(f"\nAll files saved in: {OUTPUT_DIR}")

# ==========================================
# 9. Identify companies with multiple PERMNOs
# ==========================================

company_securities = (
    stocks_df.groupby("permco")
    .agg(
        number_of_permnos=("permno", "nunique"),
        company_name=("company_name", "last")
    )
    .reset_index()
)

multiple_securities = company_securities[
    company_securities["number_of_permnos"] > 1
].sort_values(
    "number_of_permnos",
    ascending=False
)

print("\n========== MULTIPLE PERMNO COMPANIES ==========")

print(
    f"Companies with multiple PERMNOs: "
    f"{len(multiple_securities):,}"
)

display(multiple_securities.head(20))


Files will be saved to: /home/uwaterloo/nzanatta/A2
Loading library list...
Done


/tmp/40500376.1.jupyterhub.q/ipykernel_513996/2625214366.py:98: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  stocks_df["date"] = (
/tmp/40500376.1.jupyterhub.q/ipykernel_513996/2625214366.py:104: FutureWarning: ChainedAssignmentError: behav


Saved: /home/uwaterloo/nzanatta/A2/sp500_stocks_2000_2025.csv
Rows: 156,560


/home/uwaterloo/nzanatta/A2/sp500_stocks_2000_2025.csv


Saved: /home/uwaterloo/nzanatta/A2/market_returns_2000_2025.csv
Rows: 312


...

,date,market_return
0,2000-01-31,-0.0433
1,2000-02-29,0.0289
2,2000-03-31,0.0568
3,2000-04-30,-0.0593
4,2000-05-31,-0.0389
5,2000-06-30,0.0508
6,2000-07-31,-0.0200
7,2000-08-31,0.0753
8,2000-09-30,-0.0493
9,2000-10-31,-0.0222



========== RISK-FREE RATE ==========


,date,rf_1m
0,2000-01-31,0.0041
1,2000-02-29,0.0043
2,2000-03-31,0.0047
3,2000-04-30,0.0046
4,2000-05-31,0.005
5,2000-06-30,0.004
6,2000-07-31,0.0048
7,2000-08-31,0.005
8,2000-09-30,0.0051
9,2000-10-31,0.0056



========== SUMMARY ==========
Stock observations: 156,560
Unique securities (PERMNO): 1,082
Unique companies (PERMCO): 1,070
Market observations: 312
Risk-free observations: 312

All files saved in: /home/uwaterloo/nzanatta/A2

========== MULTIPLE PERMNO COMPANIES ==========
Companies with multiple PERMNOs: 12


,permco,number_of_permnos,company_name
243,11937,2,EXPAND ENERGY CORP
282,14105,2,SANDISK CORP
429,20569,2,DELTA AIR LINES INC
494,20935,2,STARWOOD HOTELS & REST WLDWD INC
564,21287,2,TWENTY FIRST CENTURY FOX INC
672,21833,2,SPRINT NEXTEL CORP
875,43613,2,COMCAST CORP NEW
894,45483,2,ALPHABET INC
905,47011,2,DISCOVERY INC
909,48779,2,UNDER ARMOUR INC
